# Parameter Interpretation
As with logistic regression, the trickier aspect of Poisson regression is the interpretation of the model parameters. Due to the link function, care must be taken when attempting to understand the estimated values so that no misunderstandings occur. However, because the link function for Poisson regression is much simpler, this tends to be an easier process compared to logistic regression.

## Logarithms
As indicated previously, Poisson regression most commonly uses the *log* link. As such, we need to establish some definitions and rules about logarithms before we continue. As you have hopefully come across before, the base $b$ logarithm of $x$ gives the value that $b$ must be raised to in order to equal $x$. As such, if 

$$
    \log_{b}\left(x\right) = y
$$

then

$$
    b^{y} = x.
$$

For instance $\log_{10}\left(100\right) = 2$ because $10^{2} = 100$. Similarly, $\log_{2}\left(16\right) = 4$ because $2^{4} = 16$.

The log link is based on the *natural* logarithm, which uses [Euler's number](https://en.wikipedia.org/wiki/E_(mathematical_constant)) as its base ($e \approx 2.718282$) and is sometimes written as $\ln$. As such, we have to raise $e$ to the power of the natural logarithm to get back to our original value. This is also known as the *exponential function*, defined as $\exp\left(x\right) = e^{x}$. `R` defines $e$ through the exponential function, meaning that $e$ itself is given by `exp(1)`, rather than being defined as a standalone constant (such as `pi`). 

Irrespective of the base, the main rules of logarithms that we need to know for understanding Poisson regression are:

1. $\log\left(ab\right) = \log\left(a\right) + \log\left(b\right)$
2. $\log\left(\frac{a}{b}\right) = \log\left(a\right) - \log\left(b\right)$
3. $\log\left(1\right) = 0$

In fact, the first two of these rules are the main reasons why logarithms were invented, because they turn multiplication into addition and division into subtraction. In the days before computers, this conversion made very difficult multiplication and division problems much easier through much simpler arithmetic. Tables of conversions for logarithms were published to allow complex problems to be expressed as logarithms, solved, and then converted back again. Obviously, this is somewhat redundant now thanks to computers. However, the logarithmic scale has many properties that are useful beyond this original intent [^foot-1].

We will see how these rules are used when exploring the meaning of the Poisson regression parameters below.

[^foot-1]: Maximum likelihood estimation is a good example of this. Even though this is solved by a computer, multiplying together many small probabilities can cause big problems with numeric precision. Instead, expressing this on the log scale not only allows these values to be *added*, but also places them on a larger scale that is less vulnerable to problems of precision. This is why the *log likelihood* is used for parameter estimation.


## Simple Poisson Regression Interpretation
In terms of interpreting the Poisson regression parameters, we will start by considering simple Poisson regression, consisting of only an intercept $\left(\beta_{0}\right)$ and a slope $\left(\beta_{1}\right)$

$$
    \log\left(\lambda_{i}\right) = \beta_{0} + \beta_{1}x_{i}.
$$


### The Intercept
In order to interpret the intercept, we can isolate its parameter by setting $x = 0$. This gives

$$
    \log\left(\lambda_{x=0}\right) = \beta_{0}
$$

This tells us that the intercept is the *log average count when all the predictors are 0*. To make this easier to understand, we can undo the log transformation using 

$$
    e^{\beta_{0}} = \lambda_{x=0},
$$

giving us the *average count when all predictors are 0*. As with other forms of regression, the usefulness of this estimate will depend upon the scaling of the predictors. If a value of 0 is meaningful across all the predictors, then this becomes a potentially useful baseline. However, if 0 is not a meaningful or practical value, as in the case of `age`, then this value is less useful and only serves as a necessary part of the model structure that can be ignored for the purpose of inference. 

### The Slope
Now that we have a definition for $\beta_{0}$, we can work on isolating the slope $\beta_{1}$. Inserting the definition of $\beta_{0}$ into the model, we get

$$
    \log\left(\lambda_{i}\right) = \overbrace{\log\left(\lambda_{x=0}\right)}^{\beta_{0}} + \beta_{1}x_{i}.
$$

If we then set $x = 1$, we get

$$
    \log\left(\lambda_{x=1}\right) = \log\left(\lambda_{x=0}\right) + \beta_{1},
$$

which we can rearrange to get

$$
    \beta_{1} = \log\left(\lambda_{x=1}\right) - \log\left(\lambda_{x=0}\right). 
$$

From the rules of logs discussed above (and in the same vein as we saw for logistic regression), subtracting two logs is the same as the log of their ratio, giving

$$
    \beta_{1} = \log\left(\frac{\lambda_{x=1}}{\lambda_{x=0}}\right). 
$$

As such, the slope parameter represents the log of a *count ratio*, which we can exponentiate to get 

$$
    e^{\beta_{1}} = \frac{\lambda_{x=1}}{\lambda_{x=0}}.
$$

#### Count Ratios
As shown above, we can get the slope parameters to the point where we can interpret them as a *count ratio*. Helpfully, this is much the same as an odds ratio from the perspective of interpretation. If $e^{\beta_{1}} = 1$ then the predicted average count does not change as the value of $x$ increases. If $e^{\beta_{1}} > 1$ then the predicted average count *increases* as the value of $x$ increases, and if $e^{\beta_{1}} < 1$ then the predicted average count *decreases* as the value of $x$ increases.

Perhaps most importantly, this is a *multiplicative* change in the average count. We can see this by starting from the definition of the count ratio

$$
    e^{\beta_{1}} = \frac{\lambda_{x=1}}{\lambda_{x=0}},
$$

and can then solve for $\lambda_{x=1}$ to give

$$
    \lambda_{x=1} = \lambda_{x=0} \times e^{\beta_{1}}.
$$

This tells us that the average count when $x = 1$ is equal to the average count when $x=0$ multiplied by the count ratio. 

As an example, if the expected count at $x=0$ is 10 and $e^{\beta_{1}} \approx 1.49$, then and we have

$$
    \lambda_{x=1} = 10 \times 1.49 = 14.9,
$$

which is a 49% increase from the baseline of 10 [^foot-2]. If we then increase $x$ from 1 to 2, we get $e^{\beta_{1} \times 2} \approx 2.22$. This gives an expected average count at $x=2$ of

$$
    \lambda_{x=2} = 10 \times 2.22 = 22.2,
$$

which is a 49% increase from 14.9 [^foot-3]. As such, each unit increase in $x$ is associated with a 49% increase in the average count, relative to the previous average count.

[^foot-2]: Importantly, this is taking 49% of 10 and then adding it on. $0.49 \times 10 = 4.9$ meaning $10 + 4.9 = 14.9$. So the increase is described based upon a certain percentage of the previous value. This is the key conceptual point. 

[^foot-3]: Again, $0.49 \times 14.9 = 7.3$, giving $14.9 + 7.3 = 22.2$. So this is a change equivalent to 49.% of the previous value.


#### Count Ratios as Percentage Change
In terms of interpreting the actual numeric value we can, as shown above, convert ratios into a *percentage change*. This is exactly the same as with odds ratios. However, because we do not have the potential confusion with probability, this can be a less error-prone move that does allow greater ease of interpretation. As such, we can use

$$
\text{\% change} = \left(e^{\beta_{1}} - 1\right) \times 100.
$$

For instance, using the same example from above of $e^{\beta_{1}} = 1.49$ then $(1.49 - 1) \times 100 = 49\%$, indicating a 49% *increase* in the average count for each unit increase of $x$. Similarly, if $e^{\beta_{1}} = 0.49$ then $(0.49 - 1) \times 100 = -51\%$, indicating a 51% *decrease* in the average count for each unit increase of $x$.

Hopefully it is clear why the relationship is therefore not linear on the count scale. As the predicted count gets larger, so too will the percentage value. 50% of 10 is obviously much smaller than 50% of 50, even though the percentage taken remains constant. As such the prediction on the count scale will get exponentially larger or smaller. This also means that the comparison of average counts will depend upon the values chosen for $x$, precisely because the relationship is an exponential curve, rather than a straight line.

## Multiple Poisson Regression Interpretation
In the case of multiple Poisson regression, the interpretation of the intercept remains the same. However, for individual slopes, calculating predicted counts for a single predictor requires a choice in terms of what to set the value of the other predictor to. This is exactly the same as we saw previously for logistic regression and as such can again use the `effects` package approach of holding all other variables at their *mean*. 

For instance, if our count was of the number of items correctly remembered during 5 minutes of free recall, we might have *age*, *sex* and *sleep duration* in the model. The predicted average count at age 50 would be

$$
    \lambda_{\text{age}=50} = e^{\beta_{0} + (\beta_{1} \times 50) + (\beta_{2} \times \text{sex}) + (\beta_{3} \times \text{sleep})},
$$

which we could calculate by holding sex and sleep at their average values.

:::{tip .simple icon=false} What You Now Know
In this section, we have discussed the interpretation of the parameters from Poisson regression. After reading this section, you should have a good sense of:
- ...
:::